In [0]:
use catalog identifier(:catalog);
use schema identifier(:schema);

In [0]:
declare or replace qry_str string;

set var qry_str = 
  "create or replace view metv_usage
  with metrics
  language yaml
  as $$
  version: 1.1
  comment: DBSQL warehouse usage metrics by warehouse and date.
  
  source: " || :catalog || '.' || :schema || ".fct_usage
  
  joins:
    - name: calendar
      source: " || :catalog || '.' || :schema || ".dim_calendar
      on: calendar.calendar_key = source.calendar_key

    - name: compute
      source: " || :catalog || '.' || :schema || ".dim_compute
      on: compute.compute_key = source.compute_key

  
  dimensions:
    - name: Date
      expr: calendar.calendar_date
      display_name: Usage Date
      comment: Date of the usage record, this field can be used for faster aggregation by date.
      synonyms:
        - usage date
        - billing date
        - consumption date
    - name: Warehouse Name
      expr: compute.warehouse_name
      display_name: Warehouse
      comment: The name of the SQL warehouse.
      synonyms:
        - warehouse
        - endpoint
        - SQL warehouse
    - name: Usage Unit
      expr: usage_unit
      display_name: Unit
      comment: Unit this usage is measured in. Possible values include DBUs.
      synonyms:
        - unit of measure
        - billing unit
        - DBU
    - name: Usage Type
      expr: usage_type
      display_name: Usage Type
      comment: The type of usage attributed to the product or workload for billing purposes. Possible values are COMPUTE_TIME, STORAGE_SPACE, NETWORK_BYTES, API_CALLS, TOKEN, or GPU_TIME.
      synonyms:
        - consumption type
        - billing type
        - workload type
    - name: Billing Origin Product
      expr: billing_origin_product
      display_name: Product
      comment: The product that originated the usage. Some products can be billed as different SKUs.
      synonyms:
        - product
        - origin product
        - billing product
    - name: SKU Name
      expr: sku_name
      display_name: SKU
      comment: Name of the SKU.
      synonyms:
        - SKU
        - pricing tier
        - billing SKU

  measures:
    - name: Usage Sum
      expr: SUM(usage_quantity)
      display_name: Total Usage
      comment: Number of units consumed for this record.
      synonyms:
        - total consumption
        - DBU usage
        - usage quantity
  $$";

execute immediate qry_str;